# Laboratorio 17 — Diez años de verdad

**Duración: 45 minutos.**

Todo lo que has hecho en este curso fue sobre tablas de treinta mil documentos. Con ese
tamaño **todo responde al instante**, y eso tiene un problema: nunca se vio la diferencia
entre hacer las cosas bien y hacerlas mal. Las dos se sienten igual de rápidas.

Hoy la bodega tiene **diez millones de documentos y diez años de historia**, y las mismas
preguntas se hacen con el reloj a la vista.

---

### Las tres mediciones de hoy

1. **Leer solo el mes que se pide**, frente a dar vuelta la bodega entera.
2. **Contar sin abrir un solo papelito**, preguntándole a la portada de la libreta.
3. **Lo que cuesta tener muchos papelitos chicos**, antes y después de pasarlos en limpio.

### Cómo se mide el tiempo

Las celdas que miden llevan **dos líneas** arriba del SQL:

```
%%time
%%sql
```

`%%time` es un cronómetro: cuando la celda termina, imprime debajo del resultado cuánto
demoró. La línea que importa es la última, **`Wall time`**, que es el tiempo de reloj de
pared, el que habrías medido tú mirando el segundero. Las otras dos dicen cuánto trabajó
el procesador de este cuaderno, que no es lo que estamos comparando.

Las celdas ya vienen con las dos líneas escritas. No hay que teclearlas.

### Una regla de hoy

**No escribes nada en estas tablas.** Son compartidas: las mismas dos tablas las está
consultando toda la sala al mismo tiempo. Todo lo de hoy es leer y medir.


## Paso 0 — La bodega de diez años

En el almacén de la esquina, la libreta del mostrador tenía las ventas del mes. La
bodega de atrás tiene **diez años**, y está ordenada en estantes: uno por año, y dentro
de cada estante, un cajón por mes.

```
                    LA BODEGA, DIEZ ANOS DE DOCUMENTOS

   2015         2016         2017                2023         2024
 +---------+  +---------+  +---------+        +---------+  +---------+
 | ene feb |  | ene feb |  | ene feb |        | ene feb |  | ene feb |
 | mar abr |  | mar abr |  | mar abr |  ...   | mar abr |  | mar abr |
 | may jun |  | may jun |  | may jun |        | may jun |  | may jun |
 |   ...   |  |   ...   |  |   ...   |        |   ...   |  |   ...   |
 | nov dic |  | nov dic |  | nov dic |        | nov dic |  | nov dic |
 +---------+  +---------+  +---------+        +---------+  +---------+
  1.000.000    1.000.000    1.000.000          1.000.000    1.000.000

        diez estantes  x  doce cajones  =  120 cajones
        diez millones de documentos
```

Hoy trabajas con **dos libretas que tienen exactamente los mismos diez millones de
documentos**, y se diferencian en una sola cosa: cómo están repartidos los papelitos.

| Libreta | Cómo guarda los papelitos |
|---|---|
| `curso.dte_10_anios` | Todos revueltos. Cada papelito trae documentos de los diez años |
| `curso.dte_10_anios_por_mes` | Un cajón por mes. Cada papelito trae un solo mes |

Las dos pesan casi lo mismo y tienen la misma cantidad de papelitos. Vas a ver que eso
no las hace equivalentes.

---

Antes de medir nada, **dos celdas de calentamiento, una por libreta**. No se
cronometran, y tienen su razón.

La primera consulta de la sesión enciende el motor: reserva memoria y se conecta al
catálogo. Y la primera vez que se toca una libreta hay que descolgarla y leer su portada.
Ese trabajo se hace una sola vez y después queda a mano, así que si midiéramos la primera
consulta estaríamos midiendo el arranque y no la pregunta.

Las dos celdas preguntan algo pequeño: cuántos documentos entraron un día cualquiera.
Empecemos por la libreta revuelta.

In [ ]:
%%sql
-- Celda 0.1



Y ahora la otra, para que las dos queden despiertas y la comparación sea limpia.

In [ ]:
%%sql
-- Celda 0.2



Las dos libretas despiertas y con los mismos documentos ese día.

Ahora sí, la primera medición, y es la más sorprendente de las tres.

**¿Cuántos documentos hay?** La forma obvia de preguntarlo sería contarlos. Pero la
libreta trae una **portada** donde está anotado, papelito por papelito, cuántas filas
tiene cada uno. Sumar esa columna responde la pregunta **sin abrir un solo papelito**.

Esa portada se consulta con `.files`, que se nombra con las tres partes: el espacio, la
tabla y la vista. Cada fila de `.files` es un papelito, y `record_count` es cuántos
documentos trae.

In [ ]:
%%time
%%sql
-- Celda 0.3



**Diez millones**, y en una fracción de segundo. No se leyó ni un documento: se leyó
la portada.

La misma pregunta a la otra libreta, la de los cajones por mes.

In [ ]:
%%time
%%sql
-- Celda 0.4



## Paso 1 — Contar la bodega entera

Los mismos diez millones. Las dos libretas tienen lo mismo.

Las dos celdas anteriores leyeron la portada a propósito. Ahora hagamos la pregunta como
la escribiría cualquiera, sin saber que la portada existe: **`count(*)`**.

Diez millones de documentos que contar. Mira el reloj.

In [ ]:
%%time
%%sql
-- Celda 1.1



**El mismo número, y en el mismo puñado de milisegundos.**

Esto sorprende, porque uno esperaría que contar diez millones de cosas cueste. No costó,
y la razón es que **el motor tampoco los contó**: reconoció que la pregunta era el total
de filas, fue a la portada, sumó lo que ya estaba anotado papelito por papelito y
respondió. Es exactamente lo que hiciste a mano en las dos celdas anteriores, solo que
aquí lo hizo solo.

Esto es lo que mide **tu** plataforma, Iceberg 1.3.0 sobre Spark 3.3.4. Si la próxima
versión lo resolviera de otra manera, el número cambiaría; por eso se mide en vez de
suponerlo, que es la costumbre que vale la pena llevarse de este paso.

**Ojo con lo que NO se acaba de demostrar.** Que contar sea gratis no quiere decir que
leer sea gratis. La portada sabe *cuántas* filas hay en cada papelito, pero no sabe
*qué dice* cada fila. En cuanto la pregunta necesite mirar los montos, hay que abrir
papelitos — y ahí es donde las dos libretas dejan de parecerse. Eso es el paso 2.

## Paso 2 — Junio de 2020

Ahora la pregunta que de verdad se hace todos los meses: **cuánto suma un mes**.

Junio de 2020. Un mes de ciento veinte.

Primero sobre la libreta revuelta.

In [ ]:
%%time
%%sql
-- Celda 2.1



Anota ese tiempo. Ahora **la misma pregunta, palabra por palabra**, sobre la libreta
de los cajones por mes.

In [ ]:
%%time
%%sql
-- Celda 2.2



**El mismo total, y muchísimo más rápido.**

Aquí está el laboratorio entero, y conviene entender exactamente por qué.

Cada papelito de una libreta Iceberg lleva un **rótulo** pegado: el valor mínimo y el
máximo de cada columna dentro de ese papelito. Para `fecha_emision`, el rótulo dice
"desde tal día hasta tal otro". El motor lee los rótulos —que están en la portada, no en
los datos— y descarta los papelitos que no pueden contener lo que busca.

**En la libreta de los cajones por mes**, el papelito de junio de 2020 tiene un rótulo
que dice "del 1 al 30 de junio de 2020". Los otros ciento diecinueve dicen otros meses.
El motor descarta **119 cajones mirando solo los rótulos** y abre uno.

**En la libreta revuelta**, cada papelito trae documentos de los diez años, así que el
rótulo de todos dice "desde enero de 2015 hasta diciembre de 2024". Junio de 2020 cae
dentro de ese rango en **todos**. Ningún rótulo permite descartar nada, y hay que
abrirlos todos y revisar fila por fila.

Los dos papeles pesan lo mismo. La diferencia no está en cuánto hay, sino en **cómo está
agrupado**.

---

### La regla que te llevas

No es "particiona siempre". Particionar tiene su costo y no toda tabla lo necesita.

La regla es **medir**: la misma pregunta, con el reloj, sobre los dos diseños, antes de
decidir. Con treinta mil filas los dos parecen buenos. Con diez millones, uno sirve y el
otro no.

## Paso 3 — Cuánto pesa cada una

Queda una pregunta natural: si una responde trece veces más rápido, **¿cuánto más
espacio cuesta?**

Vamos a mirarlo en el almacenamiento, no en la base de datos. Estas dos celdas **no
llevan `%%sql`**: no son consultas, son órdenes del sistema, y por eso empiezan con `!`.
Ya vienen escritas.

`du` quiere decir *disk usage*, el espacio que ocupa una carpeta.

In [ ]:
# Celda 3.1
!hdfs dfs -du -s -h /warehouse/iceberg/curso.db/dte_10_anios

Salen **dos números**. El primero es lo que pesan los archivos. El segundo es lo que
el sistema de archivos reserva contando las copias de respaldo que pide al guardar: aquí
pide tres, y como este laboratorio tiene un solo servidor de datos, la copia real es una.
**El número que importa es el primero.**

Ahora la otra.

In [ ]:
# Celda 3.2
!hdfs dfs -du -s -h /warehouse/iceberg/curso.db/dte_10_anios_por_mes

**Pesan casi lo mismo.** Unos pocos por ciento de diferencia, y a favor de la
particionada.

Eso es lo que hay que entender del paso: **ordenar no cuesta espacio**. Son los mismos
diez millones de documentos, la misma cantidad de papelitos, el mismo peso. Lo único
distinto es que en una los documentos del mismo mes están juntos y en la otra están
repartidos.

Si acaso pesa un poco menos, y hay una razón: cuando las filas parecidas quedan juntas,
se comprimen mejor. Documentos del mismo mes se parecen entre sí más que documentos
tomados al azar de diez años.

## Paso 4 — Muchos papelitos chicos

El tercer fenómeno, y para este hace falta otra tabla: una que esté **fragmentada**, con
muchos papelitos chicos. La arma el script `bin/reiniciar-lab.sh 17`, que ya corriste si
seguiste el README.

La tabla `mi_espacio.dte_un_anio_fragmentado` tiene un año de documentos, un millón, cargados con **una escritura por
día**: 366 días, 366 páginas y más de mil cuatrocientos papelitos.

Antes de tocar nada, cuántos papelitos hay.

In [ ]:
%%sql
-- Celda 4.1



Más de mil cuatrocientos papelitos para un millón de documentos: unos setecientos
documentos por papelito. Diminutos.

Antes de cronometrar nada, **una celda de calentamiento**, por la misma razón que en el
paso 0: esta es la primera vez que se leen datos de esta tabla en la sesión, y esa primera
lectura paga el arranque. Sin esta celda, el "antes" sale inflado y la comparación con el
"después" mide el arranque además de la fragmentación. No la cronometramos.

In [ ]:
%%sql
-- Celda 4.2



Ahora sí, la consulta del año, con el reloj. **Este es el número de antes.**

In [ ]:
%%time
%%sql
-- Celda 4.3



Ese es el costo de abrir mil cuatrocientas veces.

Ahora se pasan en limpio. Una sola sentencia, y demora unos veinte segundos: está
reescribiendo un millón de documentos. La salida dice cuántos papelitos leyó y cuántos
dejó en su lugar.

In [ ]:
%%time
%%sql
-- Celda 4.4



Leyó mil cuatrocientos sesenta y cuatro y escribió uno.

Cuántos quedan.

In [ ]:
%%sql
-- Celda 4.5



**Un solo papelito** para el millón de documentos, y ocupando menos bytes que los
mil cuatrocientos sesenta y cuatro de antes: al juntarlos se comprimen mejor.

Y la misma consulta de recién. **Este es el número de después.**

In [ ]:
%%time
%%sql
-- Celda 4.6



Pon los dos tiempos uno al lado del otro, que es lo que hay que ver.

Y ahora lo que casi nadie espera. Acabas de dejar el millón de documentos en **un**
papelito, así que parece que el disco debería haber bajado. Pregúntaselo a la libreta:
esta consulta pone al lado lo que la tabla **usa hoy** y lo que **sigue guardado**.

`.files` son los papelitos vigentes, los que la página de hoy nombra. `.all_files` son
los de todas las páginas, las de hoy y las de antes. Van con `DISTINCT file_path` porque
un mismo papelito aparece una vez por cada manifiesto que lo nombra, y lo que se quiere
contar son archivos, no menciones.

In [ ]:
%%sql
-- Celda 4.7



**El disco no bajó: subió.** La tabla usa un papelito, pero en el disco están los mil
cuatrocientos sesenta y cuatro viejos **más** el nuevo, y entre todos pesan cerca del
doble que antes de compactar.

No es una falla. Las páginas viejas de la libreta todavía nombran esos papelitos, y
mientras existan se puede pedir la tabla como estaba ayer. Compactar **no borra nada**:
escribe la hoja en limpio y deja de usar las viejas.

El espacio se libera recién cuando se botan esas páginas, con `expire_snapshots`, y esa
es una decisión aparte y deliberada: cuánta historia se conserva no lo decide quien
compacta, lo decide la exigencia de auditoría de la institución.

---

## Listo

Tres mediciones, con el reloj a la vista y diez millones de documentos abajo.

| Lo que se preguntó | Libreta revuelta | Libreta por mes | Qué tuvo que abrir |
|---|---|---|---|
| Cuántos documentos hay, por la portada | décimas de segundo | décimas de segundo | nada |
| Cuántos documentos hay, con `count(*)` | décimas de segundo | décimas de segundo | nada |
| **Cuánto suma junio de 2020** | **segundos** | **décimas de segundo** | 120 papelitos contra 1 |
| Cuánto pesa en el almacenamiento | 246,5 M | 227,0 M | — |

Tus tiempos exactos están en tus celdas y no van a ser idénticos a los del de al lado:
la sala consulta las mismas dos tablas a la vez. Lo que sí es igual para todos es la
**proporción** de la tercera fila, que es la que importa.

Y el tercer fenómeno, el que proyectó el instructor: más de mil cuatrocientos papelitos
chicos cuestan caro, pasarlos en limpio los deja en unos pocos, y el espacio no baja
hasta que se botan las páginas viejas.

### La idea de fondo

**Con treinta mil filas, cualquier diseño parece bueno.** Todo responde al instante y no
hay manera de distinguir una tabla bien pensada de una mal pensada.

Con diez millones, la diferencia entre un diseño y otro es la diferencia entre **un
reporte que sale y uno que no sale**. Y la única forma de saber de qué lado estás es la
que usaste hoy: hacer la misma pregunta sobre los dos diseños, con el reloj, antes de
decidir.

Las tablas con las que vas a trabajar no tienen treinta mil filas.
